In [2]:
!pip install mlflow

import os
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# SETTINGS
# ============================================================

DATA_PATH = "/content/aiml_training_data.csv"

# CHANGE THIS if your target column has a different name
TARGET_COLUMN = "intent"

MODEL_DIR = "models"
OUTPUT_DIR = "outputs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# LOAD DATA
# ============================================================

print("\nLoading processed dataset...")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))


# ============================================================
# CHECK TARGET
# ============================================================

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"\nTarget column '{TARGET_COLUMN}' was not found.\n"
        f"Available columns are:\n{list(df.columns)}\n"
        "Change TARGET_COLUMN in train_mlflow.py."
    )


# ============================================================
# FEATURES AND TARGET
# ============================================================

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("\nTarget distribution:")
print(y.value_counts())


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# ============================================================
# IDENTIFY DATA TYPES
# ============================================================

numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)


# ============================================================
# PREPROCESSING
# ============================================================

numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])


categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])


preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_columns
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_columns
    )
])


# ============================================================
# MLFLOW
# ============================================================

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("file:./mlruns")

mlflow.set_experiment(
    "credit-risk-model-tuning"
)


# ============================================================
# HYPERPARAMETER CONFIGURATIONS
# ============================================================

configs = [

    {
        "name": "configuration_1",
        "C": 0.1,
        "solver": "liblinear",
        "class_weight": None
    },

    {
        "name": "configuration_2",
        "C": 1.0,
        "solver": "liblinear",
        "class_weight": "balanced"
    },

    {
        "name": "configuration_3",
        "C": 10.0,
        "solver": "liblinear",
        "class_weight": None
    }

]


# ============================================================
# EXPERIMENT RESULTS
# ============================================================

results = []

best_f1 = -1
best_model = None
best_config = None


# ============================================================
# TRAIN EACH CONFIGURATION
# ============================================================

for config in configs:

    print("\n")
    print("=" * 60)
    print("Running:", config["name"])
    print("=" * 60)

    with mlflow.start_run(
        run_name=config["name"]
    ):

        # ----------------------------------------------------
        # MODEL
        # ----------------------------------------------------

        classifier = LogisticRegression(
            C=config["C"],
            solver=config["solver"],
            class_weight=config["class_weight"],
            max_iter=1000,
            random_state=42
        )


        model = Pipeline([
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                classifier
            )
        ])


        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        model.fit(
            X_train,
            y_train
        )


        # ----------------------------------------------------
        # PREDICT
        # ----------------------------------------------------

        predictions = model.predict(
            X_test
        )


        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        precision = precision_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        recall = recall_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )


        # ----------------------------------------------------
        # LOG PARAMETERS
        # ----------------------------------------------------

        mlflow.log_param(
            "model",
            "LogisticRegression"
        )

        mlflow.log_param(
            "C",
            config["C"]
        )

        mlflow.log_param(
            "solver",
            config["solver"]
        )

        mlflow.log_param(
            "class_weight",
            str(config["class_weight"])
        )

        mlflow.log_param(
            "test_size",
            0.20
        )

        mlflow.log_param(
            "random_state",
            42
        )


        # ----------------------------------------------------
        # LOG METRICS
        # ----------------------------------------------------

        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )


        # ----------------------------------------------------
        # LOG MODEL
        # ----------------------------------------------------

        mlflow.sklearn.log_model(
            model,
            "model",
            skops_trusted_types=[
                "numpy.dtype",
                "sklearn.compose._column_transformer._RemainderColsList",
            ]
        )


        # ----------------------------------------------------
        # SAVE RESULT
        # ----------------------------------------------------

        results.append({

            "configuration": config["name"],

            "C": config["C"],

            "solver": config["solver"],

            "class_weight": config["class_weight"],

            "accuracy": accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1

        })


        # ----------------------------------------------------
        # DISPLAY
        # ----------------------------------------------------

        print(
            f"Accuracy : {accuracy:.4f}"
        )

        print(
            f"Precision: {precision:.4f}"
        )

        print(
            f"Recall   : {recall:.4f}"
        )

        print(
            f"F1 Score : {f1:.4f}"
        )


        # ----------------------------------------------------
        # CHECK BEST MODEL
        # ----------------------------------------------------

        if f1 > best_f1:

            best_f1 = f1

            best_model = model

            best_config = config.copy()


# ============================================================
# SAVE EXPERIMENT RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)

results_path = os.path.join(
    OUTPUT_DIR,
    "mlflow_experiment_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)


# ============================================================
# SAVE BEST MODEL
# ============================================================

best_model_path = os.path.join(
    MODEL_DIR,
    "best_model.joblib"
)

joblib.dump(
    best_model,
    best_model_path
)


# ============================================================
# SAVE BEST CONFIGURATION
# ============================================================

best_config_path = os.path.join(
    OUTPUT_DIR,
    "best_configuration.txt"
)

with open(
    best_config_path,
    "w"
) as file:

    file.write(
        "BEST MODEL CONFIGURATION\n"
    )

    file.write(
        "=========================\n\n"
    )

    for key, value in best_config.items():

        file.write(
            f"{key}: {value}\n"
        )

    file.write(
        f"\nBest F1 Score: {best_f1:.4f}\n"
    )


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n")
print("=" * 60)
print("EXPERIMENT COMPARISON")
print("=" * 60)

print(
    results_df.to_string(index=False)
)


print("\n")
print("=" * 60)
print("BEST CONFIGURATION")
print("=" * 60)

print(
    f"Configuration : {best_config['name']}"
)

print(
    f"C             : {best_config['C']}"
)

print(
    f"Solver        : {best_config['solver']}"
)

print(
    f"Class Weight  : {best_config['class_weight']}"
)

print(
    f"Best F1 Score  : {best_f1:.4f}"
)


print("\nSaved files:")

print(
    f"- {best_model_path}"
)

print(
    f"- {results_path}"
)

print(
    f"- {best_config_path}"
)

print("\nCheckpoint 3 training complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

2026/08/30 13:56:20 INFO mlflow.tracking.fluent: Experiment with name 'credit-risk-model-tuning' does not exist. Creating a new experiment.



Loading processed dataset...
Dataset shape: (230, 3)
Columns: ['text', 'intent', 'confidence']

Target distribution:
intent
return_request    46
refund_enquiry    42
complaint         39
delivery_delay    39
product_query     34
order_status      30
Name: count, dtype: int64

Training samples: 184
Testing samples: 46

Numerical columns:
['confidence']

Categorical columns:
['text']


Running: configuration_1


2026/08/30 13:56:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/30 13:56:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy : 0.1957
Precision: 0.0383
Recall   : 0.1957
F1 Score : 0.0640


Running: configuration_2


2026/08/30 13:56:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy : 0.1957
Precision: 0.0383
Recall   : 0.1957
F1 Score : 0.0640


Running: configuration_3
Accuracy : 0.1957
Precision: 0.0383
Recall   : 0.1957
F1 Score : 0.0640


EXPERIMENT COMPARISON
  configuration    C    solver class_weight  accuracy  precision   recall  f1_score
configuration_1  0.1 liblinear         None  0.195652    0.03828 0.195652  0.064032
configuration_2  1.0 liblinear     balanced  0.195652    0.03828 0.195652  0.064032
configuration_3 10.0 liblinear         None  0.195652    0.03828 0.195652  0.064032


BEST CONFIGURATION
Configuration : configuration_1
C             : 0.1
Solver        : liblinear
Class Weight  : None
Best F1 Score  : 0.0640

Saved files:
- models/best_model.joblib
- outputs/mlflow_experiment_results.csv
- outputs/best_configuration.txt

Checkpoint 3 training complete.
